In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../..')
import gc
import json
import os
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from Model.Combined.DataPrep.Data_Prep import Combined_Data_Prep
from Model.Combined.DataPrep.Player_Dataset import Create_Test_Train_Datasets
from Model.Pro.DataPrep import Prep_Map, Output_Map
from Model.College.DataPrep import Prep_Map as C_Prep_Map, Output_Map as C_Output_Map
from Model.Pro.Model.Player_Model import Recurrent_Model as Pro_Model
from Model.College.Model.College_Model import RNN_Model as College_Model
from Model.Combined.Model.Model_Train import TrainAndGraph
from Model.Constants import device, NUM_MODEL_VARIANTS

In [ ]:
is_hitter = True

CUTOFFS = [2, 3, 4, 5, 7, 10, 15, 20, 30, 40, 50, 70, 100]
NUM_REPEATS = 4
NUM_RUNS = NUM_REPEATS * NUM_MODEL_VARIANTS

RESULTS_FILE = "mean_revert_sweep_results.json"
BASELINE_KEY = "baseline"

Short Helpers

In [ ]:
def CutoffLabel(cutoff: float) -> str:
    return f"cutoff_{cutoff:g}"

def LoadResults() -> dict:
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, "r") as f:
            return json.load(f)
    return {}

def SaveResults(results: dict):
    with open(RESULTS_FILE, "w") as f:
        json.dump(results, f, indent=2)
        
def FreeCudaMemory():
    gc.collect()
    torch.cuda.empty_cache()

Training Helpers

In [ ]:
def RunSingleVariant(data_prep, io_list, train_idx: int) -> float:
    """Train one model variant and return best_loss_war, aggressively freeing memory afterwards."""
    train_dataset, test_dataset = Create_Test_Train_Datasets(
        player_list=io_list,
        is_hitter=is_hitter,
        train_idx=train_idx)
    
    pro_model = Pro_Model(
        input_size=data_prep.GetProIOSize(is_hitter),
        data_prep=data_prep.pro_data_prep,
        is_hitter=is_hitter,
    ).to(device)
    
    col_model = College_Model(
        input_size=data_prep.GetColIOSize(is_hitter),
        data_prep=data_prep.college_data_prep,
        is_hitter=is_hitter,
        output_init_state_size=pro_model.GetInitStateSize(),
    ).to(device)
    
    train_results = TrainAndGraph(
        pro_network=pro_model,
        col_network=col_model,
        train_dataset=train_dataset,
        test_dataset=test_dataset,
        pro_model_name="../Models/test_pro",
        col_model_name="../Models/test_col",
        is_hitter=is_hitter,
        should_output=False,
        show_progress_bar=False
    )
    best_loss_war = train_results.best_loss_war
    
    # Manual cleanup: datasets/models are large, don't wait for GC.
    del train_results
    del col_model
    del pro_model
    del test_dataset
    del train_dataset
    FreeCudaMemory()
    
    # Re-run if bad training
    if (is_hitter and best_loss_war > 8.5) or (not is_hitter and best_loss_war > 9):
        return RunSingleVariant(data_prep, io_list, train_idx)
    
    return best_loss_war

In [ ]:
def RunPrepMapVariants(pro_prep_map, key: str, results: dict):
    """Runs all variants for one pro prep map, storing per-variant losses in results[key]."""
    done = results.get(key, [])
    if len(done) >= NUM_RUNS:
        return
    
    data_prep = Combined_Data_Prep(
        pro_prep_map,
        Output_Map.base_output_map,
        C_Prep_Map.college_base_prep_map,
        C_Output_Map.college_output_map)
    
    io_list = data_prep.Generate_IO_Hitters(is_training=True) if is_hitter else \
        data_prep.Generate_IO_Pitchers(is_training=True)
    try:
        results[key] = done
        
        variant_iter = tqdm(
            range(len(done), NUM_RUNS),
            initial=len(done),
            total=NUM_RUNS,
            desc=key,
            leave=False)
        
        for variant_idx in variant_iter:
            loss = RunSingleVariant(data_prep, io_list, variant_idx % NUM_MODEL_VARIANTS)
            results[key].append(loss)
            SaveResults(results)
    finally:
        del io_list, data_prep
        FreeCudaMemory()
        
def SumLoss(results: dict, key: str) -> float | None:
    losses = results.get(key, [])
    if len(losses) < NUM_MODEL_VARIANTS:
        return None  # incomplete
    return sum(losses)

Run Sweep

In [ ]:
results = LoadResults()

RunPrepMapVariants(Prep_Map.base_prep_map, BASELINE_KEY, results)
baselineSum = SumLoss(results, BASELINE_KEY)

for cutoff in tqdm(CUTOFFS, desc="Cutoffs"):
    RunPrepMapVariants(Prep_Map.MakeMeanRevertPrepMap(cutoff), CutoffLabel(cutoff), results)

Plotting

In [ ]:
# Reload results so previous cell can be commented out if not required
results = LoadResults()
baselineSum = SumLoss(results, BASELINE_KEY)

In [ ]:
xs, ys = [], []
for cutoff in CUTOFFS:
    s = SumLoss(results, CutoffLabel(cutoff))
    if s is not None:
        xs.append(cutoff)
        ys.append(s)
        
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(xs, ys, marker="o", label="MakeMeanRevertPrepMap")

ax.axhline(baselineSum, color="red", linestyle="--", label="base_prep_map (baseline)")

ax.set_xscale("log")
ax.set_xlabel("Mean-revert cutoff (PA / BF for half weight)")
ax.set_ylabel(f"Sum of best_loss_war over {NUM_MODEL_VARIANTS} variants")
ax.set_title(f"Mean-revert cutoff vs. WAR loss ({'hitters' if is_hitter else 'pitchers'})")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()